## RUL CPFL — Réplica Metodológica Adaptada: Abdouni et al. (Compress-and-Flatten + Wide & Deep)

Referência: [PHME 4906](https://papers.phmsociety.org/index.php/phme/article/download/4906/2940) (§3 e §4.2) | Modelo: `amazon/chronos-2` (`29ec3766d36d6f73f0696f85560a422f50e8498c`, `chronos-forecasting==2.3.2`).

### Fidelidade e Adaptações
- **Fiel ao artigo**: prefixo observado inteiro `<= endpoint` (recusa `> 8192`, sem truncamento silencioso), `z-score` sem clipping ajustado apenas no treino, projeção compartilhada por canal `Linear(768->8)+GELU` (`d_comp=8`) + `flatten(64)` + concatenação dos `8` sensores atuais normalizados (`72` features), `MLP(256, 64, 1)` com `GELU`, `Dropout(0.2)` após cada oculta (interpretação explicitada) e saída linear (sem clamp implícito), `AdamW(lr=1e-4, weight_decay=1e-3)`, `batch_size=1024` (lote único quando `n < 1024`; `batch_size=64` configurável como adaptação manual declarada), `max_epochs=150`, `patience=15` por `validation RMSE`, `10` sementes (`42..51`).
- **Regimes omitidos**: os CSVs CPFL não possuem settings operacionais físicos (altitude/Mach/throttle); portanto `KMeans`/one-hot de regime e normalização por regime são omitidos. As 6 posições de medição do `gearbox` **não** são regimes.
- **Adaptações ao protocolo CPFL**: alvo em fração `0..1` (`porcentagem-data`), grade causal `12h` com gap máximo `48h`, validação agrupada aninhada (`gerador` 4 grupos / 574 endpoints; `gearbox` 3 eventos / 822 endpoints, independentes), sem `Cap 125`, `NASA Score` ou `MAPE`.
- **Ablações**: `fusion` (primário), `deep` (sem ramo Wide) e `wide` (controle adicional para medir o ganho real dos embeddings).


In [ ]:
# =============================================================================
# Configuracao principal (edite aqui antes de executar)
# =============================================================================
CONFIG = {
    'dataset': 'gerador',  # 'gerador' (574 endpoints) ou 'gearbox' (822 endpoints)
    'mode': 'smoke',       # 'smoke' | 'extract' | 'train' | 'full'
    'input_csv': '/content/Features_gerador_score_RUL-porcentagem-data.csv',
    'output_dir': './chronos_outputs',  # Para Drive no Colab: '/content/drive/MyDrive/cer_chronos_outputs' com mount_drive=True
    'cache_dir': None,                  # None usa <output_dir>/abdouni/<dataset>/cache
    'run_id': None,
    'device': 'cuda',      # Colab GPU padrao; 'cpu' explicito e suportado (mais lento)
    'd_comp': 8,
    'max_epochs': 150,
    'patience': 15,
    'batch_size': 1024,    # 1024 fiel ao PHME 4906; 64 pode ser usado como adaptacao manual declarada
    'lr': 1e-4,
    'weight_decay': 1e-3,
    'dropout': 0.2,
    'seeds': list(range(42, 52)),  # 10 repeticoes (42..51)
    'variants': ['fusion', 'deep', 'wide'],
    'install_dependencies': False,
    'mount_drive': False,
    'download_zip': False,
}


In [ ]:
# =============================================================================
# Ambiente Colab / Drive / Upload (somente quando habilitado ou em falta)
# =============================================================================
import os, subprocess, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if CONFIG.get('install_dependencies'):
    if Path('requirements-chronos-colab.txt').is_file():
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-chronos-colab.txt'])
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'chronos-forecasting==2.3.2', 'torch>=2.2,<3', 'transformers>=4.41,<6'])

if CONFIG.get('mount_drive'):
    if not IN_COLAB:
        raise RuntimeError('mount_drive=True requer execucao no Google Colab.')
    from google.colab import drive
    drive.mount('/content/drive')

_drive_root_raw = Path(os.path.expanduser(str(CONFIG.get('drive_mount_root') or '/content/drive')))
_drive_root = Path(os.path.abspath(str(_drive_root_raw))).resolve(strict=False)
for _key in ('output_dir', 'cache_dir', 'run_dir'):
    _val = CONFIG.get(_key)
    if _val:
        _cand = Path(os.path.abspath(os.path.expanduser(str(_val)))).resolve(strict=False)
        if (_cand == _drive_root or _cand.is_relative_to(_drive_root)):
            if not (_drive_root.is_dir() and (_drive_root / 'MyDrive').is_dir() and (os.path.ismount(str(_drive_root_raw)) or os.path.ismount(str(_drive_root)))):
                raise RuntimeError(
                    f"CONFIG[{_key!r}]={_val!r} aponta para o Google Drive, mas '{_drive_root}' nao esta montado. "
                    "Habilite CONFIG['mount_drive']=True no Colab ou use um diretorio local (ex.: './chronos_outputs')."
                )

if IN_COLAB:
    missing_files = [f for f in ('cer_rul.py', 'chronos_rul.py') if not Path(f).is_file()]
    if not Path(CONFIG['input_csv']).is_file():
        missing_files.append(CONFIG['input_csv'])
    if missing_files:
        print('Arquivos ausentes no runtime:', missing_files)
        print('Faca upload de cer_rul.py, chronos_rul.py e do CSV selecionado.')
        from google.colab import files
        files.upload()

for mod_file in ('cer_rul.py', 'chronos_rul.py'):
    if not Path(mod_file).is_file():
        raise FileNotFoundError(f'Modulo obrigatorio ausente: {mod_file}. Faca upload antes de continuar.')
if not Path(CONFIG['input_csv']).is_file():
    raise FileNotFoundError(f"CSV nao encontrado em {CONFIG['input_csv']!r}. Ajuste CONFIG['input_csv'] ou faca upload.")


In [ ]:
# =============================================================================
# Execucao via nucleo compartilhado (chronos_rul.py)
# =============================================================================
import chronos_rul

RESULT = chronos_rul.run_article_pipeline('abdouni', CONFIG, CONFIG['mode'])
print('Status:', RESULT['status'], '| Dataset:', RESULT['dataset'])
print('ZIP:', RESULT.get('zip_path'))


In [ ]:
# =============================================================================
# Relatorio de ablacoes, epocas de refit e metricas (fracao 0..1)
# =============================================================================
import json
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

if RESULT['status'] == 'smoke':
    print(json.dumps({k: RESULT[k] for k in ('status', 'dataset', 'clean_keys', 'endpoint_count', 'outer_folds')}, indent=2))
elif RESULT['status'] == 'trained':
    print('Contagem real de parametros:', RESULT['param_counts'])
    rows = []
    for variant, block in RESULT['metrics_summary'].items():
        for scale_var in ('raw', 'clipped_0_1'):
            for scope in ('macro', 'pooled'):
                m = block[scale_var][scope]
                rows.append({
                    'variant': variant, 'scale_variant': scale_var, 'scope': scope,
                    'mae': f"{m['mae']['mean']:.5f} +/- {m['mae']['std']:.5f}",
                    'rmse': f"{m['rmse']['mean']:.5f} +/- {m['rmse']['std']:.5f}",
                    'bias': f"{m['bias']['mean']:.5f} +/- {m['bias']['std']:.5f}",
                })
    display(pd.DataFrame(rows))

if CONFIG.get('download_zip') and RESULT.get('zip_path') and 'google.colab' in sys.modules:
    from google.colab import files
    files.download(RESULT['zip_path'])
